# 第二プロジェクト DB 半年前差分（高速版）

新DB (`db.zst.part0` + `db.zst.part1`) と、固定コミット
`ce1e6788c9461a550197e1027df59e575450eac8` の旧DB `comment.db.zst`
をコメントID単位で比較します。

旧DBには index が無い前提なので、比較用SQLiteへ一度だけ正規化コピーし、
`id INTEGER PRIMARY KEY` を付けてから差分を計算します。

出力: `summary.json`, `added.csv`, `removed.csv`, `changed.csv`, `db_diff.zip`


In [ ]:
!apt-get -qq update
!apt-get -qq install -y zstd
!pip -q install requests


In [ ]:
from pathlib import Path
from urllib.parse import urljoin
import csv, hashlib, json, shutil, sqlite3, subprocess, time, zipfile
import requests

NEW_BASE = 'https://raw.githubusercontent.com/22552/kasotest/main/'
OLD_DB_BLOB = 'https://git.sr.ht/~kasosuta/dai2db/blob/ce1e6788c9461a550197e1027df59e575450eac8/comment.db.zst'
OLD_DB_URL = OLD_DB_BLOB + '?raw=true'
WORK = Path('/content/dai2db-diff')
WORK.mkdir(exist_ok=True)

session = requests.Session()
session.headers['User-Agent'] = 'dai2db-colab-diff/5'


In [ ]:
def download(url, dst):
    dst = Path(dst)
    with session.get(url, stream=True, timeout=180, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length') or 0)
        done = 0
        with dst.open('wb') as f:
            for chunk in r.iter_content(1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f'\r{dst.name}: {done/2**20:.1f}/{total/2**20:.1f} MiB', end='')
    print()
    return dst

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(4 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def is_zstd(path):
    with open(path, 'rb') as f:
        return f.read(4) == b'\x28\xb5\x2f\xfd'

def zstd_decompress(src, dst):
    src, dst = Path(src), Path(dst)
    dst.unlink(missing_ok=True)
    test = subprocess.run(
        ['zstd', '-t', '--long=31', str(src)],
        text=True, capture_output=True
    )
    if test.returncode:
        print(test.stdout)
        print(test.stderr)
        raise RuntimeError('zstd整合性チェックに失敗しました')
    p = subprocess.run(
        ['zstd', '-d', '--long=31', '-f', str(src), '-o', str(dst)],
        text=True, capture_output=True
    )
    if p.returncode:
        print(p.stdout)
        print(p.stderr)
        raise RuntimeError(f'zstd展開失敗 (exit={p.returncode})')
    return dst

def is_sqlite(path):
    with open(path, 'rb') as f:
        return f.read(16) == b'SQLite format 3\x00'


In [ ]:
# 新DB
p0 = download(urljoin(NEW_BASE, 'db.zst.part0'), WORK/'db.zst.part0')
p1 = download(urljoin(NEW_BASE, 'db.zst.part1'), WORK/'db.zst.part1')

new_zst = WORK/'new.db.zst'
with new_zst.open('wb') as out:
    for p in (p0, p1):
        with p.open('rb') as f:
            shutil.copyfileobj(f, out)

print('joined:', f'{new_zst.stat().st_size/2**20:.1f} MiB')
print('sha256:', sha256(new_zst))
assert is_zstd(new_zst), '結合後ファイルがZstdではありません'

new_db = zstd_decompress(new_zst, WORK/'new.db')
assert is_sqlite(new_db), '新DBがSQLiteではありません'
print('new sqlite:', f'{new_db.stat().st_size/2**20:.1f} MiB')


In [ ]:
# 旧DB
print('old blob:', OLD_DB_BLOB)
old_zst = download(OLD_DB_URL, WORK/'comment.db.zst')
print('sha256:', sha256(old_zst))
assert is_zstd(old_zst), '旧DBの取得結果がZstdではありません'

old_db = zstd_decompress(old_zst, WORK/'old.db')
assert is_sqlite(old_db), '旧DBがSQLiteではありません'
print('old sqlite:', f'{old_db.stat().st_size/2**20:.1f} MiB')


In [ ]:
ALIASES = {
    'id': ('id','comment_id'),
    'parent_id': ('parent_id','parent','reply_to','reply_to_id'),
    'is_reply': ('is_reply',),
    'user': ('user','username','author','author_name'),
    'user_id': ('user_id','author_id'),
    'datetime': ('datetime','datetime_created','created_at','timestamp','date'),
    'content': ('content','text','body'),
}

def qi(s):
    return '"' + s.replace('"','""') + '"'

def detect_table(path):
    con = sqlite3.connect(path)
    best = None
    try:
        tables = [r[0] for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        )]
        for table in tables:
            cols = [r[1] for r in con.execute(f'PRAGMA table_info({qi(table)})')]
            lower = {c.lower(): c for c in cols}
            mapping = {}
            for logical, aliases in ALIASES.items():
                for alias in aliases:
                    if alias in lower:
                        mapping[logical] = lower[alias]
                        break
            if 'id' not in mapping:
                continue
            score = (100 if table.lower() == 'comments' else 0) + len(mapping)*10
            if best is None or score > best[0]:
                best = (score, table, mapping)
    finally:
        con.close()
    if best is None:
        raise RuntimeError(f'コメントテーブルを検出できません: {path}')
    return best[1], best[2]

new_table, new_map = detect_table(new_db)
old_table, old_map = detect_table(old_db)
print('new:', new_table, new_map)
print('old:', old_table, old_map)


In [ ]:
# 比較専用DBへ1回だけコピーし、IDをPRIMARY KEY化
CMP = WORK/'compare.db'
CMP.unlink(missing_ok=True)

con = sqlite3.connect(CMP)
con.executescript('''
PRAGMA journal_mode=OFF;
PRAGMA synchronous=OFF;
PRAGMA temp_store=MEMORY;
PRAGMA cache_size=-262144;
PRAGMA locking_mode=EXCLUSIVE;

CREATE TABLE old_norm (
  id INTEGER PRIMARY KEY,
  parent_id INTEGER,
  is_reply INTEGER,
  user TEXT,
  user_id INTEGER,
  datetime TEXT,
  content TEXT
);
CREATE TABLE new_norm (
  id INTEGER PRIMARY KEY,
  parent_id INTEGER,
  is_reply INTEGER,
  user TEXT,
  user_id INTEGER,
  datetime TEXT,
  content TEXT
);
''')
con.execute('ATTACH DATABASE ? AS olddb', (str(old_db),))
con.execute('ATTACH DATABASE ? AS newdb', (str(new_db),))

def source_select(schema, table, mapping):
    cols = []
    for logical in ALIASES:
        if logical in mapping:
            col = qi(mapping[logical])
            cast = 'INTEGER' if logical in {'id','parent_id','is_reply','user_id'} else 'TEXT'
            cols.append(f'CAST({col} AS {cast})')
        else:
            cols.append('NULL')
    return f"SELECT {', '.join(cols)} FROM {schema}.{qi(table)}"

t0 = time.time()
print('[1/2] old DBを正規化コピー + ID index化...')
con.execute('BEGIN')
con.execute('INSERT OR REPLACE INTO old_norm ' + source_select('olddb', old_table, old_map))
con.commit()
print(' old rows =', con.execute('SELECT COUNT(*) FROM old_norm').fetchone()[0],
      f'({time.time()-t0:.1f}s)')

t1 = time.time()
print('[2/2] new DBを正規化コピー + ID index化...')
con.execute('BEGIN')
con.execute('INSERT OR REPLACE INTO new_norm ' + source_select('newdb', new_table, new_map))
con.commit()
print(' new rows =', con.execute('SELECT COUNT(*) FROM new_norm').fetchone()[0],
      f'({time.time()-t1:.1f}s)')

# ここから先のJOINは両方 id PRIMARY KEY を使える
shared = [k for k in ALIASES if k != 'id' and k in new_map and k in old_map]
print('比較フィールド:', shared)


In [ ]:
# 差分を一度だけmaterialize
t0 = time.time()
print('[diff] added...')
con.execute('DROP TABLE IF EXISTS added')
con.execute('''
CREATE TABLE added AS
SELECT n.*
FROM new_norm n
LEFT JOIN old_norm o ON o.id = n.id
WHERE o.id IS NULL
''')
print(' added =', con.execute('SELECT COUNT(*) FROM added').fetchone()[0])

print('[diff] removed...')
con.execute('DROP TABLE IF EXISTS removed')
con.execute('''
CREATE TABLE removed AS
SELECT o.*
FROM old_norm o
LEFT JOIN new_norm n ON n.id = o.id
WHERE n.id IS NULL
''')
print(' removed =', con.execute('SELECT COUNT(*) FROM removed').fetchone()[0])

changed_where = ' OR '.join([f'n.{qi(k)} IS NOT o.{qi(k)}' for k in shared]) or '0'
changed_cols = ['n.id AS id']
for k in shared:
    changed_cols += [f'o.{qi(k)} AS old_{k}', f'n.{qi(k)} AS new_{k}']

print('[diff] changed...')
con.execute('DROP TABLE IF EXISTS changed')
con.execute(f'''
CREATE TABLE changed AS
SELECT {', '.join(changed_cols)}
FROM new_norm n
JOIN old_norm o ON o.id = n.id
WHERE {changed_where}
''')
print(' changed =', con.execute('SELECT COUNT(*) FROM changed').fetchone()[0])
con.commit()
print(f'[diff] done in {time.time()-t0:.1f}s')


In [ ]:
old_count = con.execute('SELECT COUNT(*) FROM old_norm').fetchone()[0]
new_count = con.execute('SELECT COUNT(*) FROM new_norm').fetchone()[0]
added_count = con.execute('SELECT COUNT(*) FROM added').fetchone()[0]
removed_count = con.execute('SELECT COUNT(*) FROM removed').fetchone()[0]
changed_count = con.execute('SELECT COUNT(*) FROM changed').fetchone()[0]

summary = {
    'old_rows': old_count,
    'new_rows': new_count,
    'delta_rows': new_count-old_count,
    'added': added_count,
    'removed': removed_count,
    'changed': changed_count,
    'compared_fields': shared,
    'old_commit': 'ce1e6788c9461a550197e1027df59e575450eac8',
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(WORK/'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)


In [ ]:
def export_table(table, path):
    cur = con.execute(f'SELECT * FROM {qi(table)} ORDER BY id')
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow([d[0] for d in cur.description])
        while True:
            rows = cur.fetchmany(10000)
            if not rows:
                break
            w.writerows(rows)

export_table('added', WORK/'added.csv')
export_table('removed', WORK/'removed.csv')
export_table('changed', WORK/'changed.csv')

with zipfile.ZipFile(WORK/'db_diff.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ('summary.json','added.csv','removed.csv','changed.csv'):
        z.write(WORK/name, arcname=name)

print('完成:', WORK/'db_diff.zip')
from google.colab import files
files.download(str(WORK/'db_diff.zip'))
